# Verify: `agent_with_helpfulness` Retry Loop

Runs a set of queries against local (`langgraph dev`) and/or production (Cloud Run),
captures `retry_count` and `is_helpful` from the final graph state, and
prints a comparison table.

**Before running:**
- For local tests: `uv run langgraph dev` in a terminal
- For prod tests: nothing extra — Cloud Run is always up

All queries for each environment run **in parallel** — total wall-clock time
is roughly the slowest single query, not the sum of all queries.

In [5]:
import asyncio
import textwrap
from dataclasses import dataclass

from dotenv import load_dotenv
from langgraph_sdk import get_client

load_dotenv()

# Make sure the URLS match deployment 
LOCAL_URL    = "http://localhost:55641" 
PROD_URL     = "https://cat-health-agent-316461491006.us-central1.run.app"
ASSISTANT_ID = "agent_with_helpfulness"

# Poll every N seconds; wait up to TIMEOUT seconds per query.
# Cloud Run can be slow on a cold start — 180 s is safe.
POLL_INTERVAL = 4.0
TIMEOUT       = 180.0

In [2]:
TEST_QUERIES = [
    "How often should I deworm my cat?",      # expect: pass first try
    "What are signs of feline dehydration?",  # expect: pass first try
    "My cat is sick, help.",                  # vague — may trigger a retry
]

In [6]:
@dataclass
class RunResult:
    query: str
    retry_count: int
    is_helpful: bool
    thread_id: str
    run_id: str | None
    error: str | None = None

    @property
    def looped(self) -> bool:
        return self.retry_count > 1


async def run_query(client, query: str) -> RunResult:
    """Create a run, poll until complete, return state fields."""
    thread = await client.threads.create()
    tid = thread["thread_id"]
    run_id = None
    try:
        run = await client.runs.create(
            tid,
            ASSISTANT_ID,
            input={"messages": [{"type": "human", "content": query}]},
        )
        run_id = run["run_id"]

        elapsed, r = 0.0, run
        while elapsed < TIMEOUT:
            await asyncio.sleep(POLL_INTERVAL)
            elapsed += POLL_INTERVAL
            r = await client.runs.get(tid, run_id)
            if r["status"] not in ("pending", "running"):
                break

        if r["status"] != "success":
            return RunResult(query=query, retry_count=0, is_helpful=False,
                             thread_id=tid, run_id=run_id,
                             error=f"status={r['status']} after {elapsed:.0f}s")

        values = (await client.threads.get_state(tid))["values"]
        return RunResult(
            query=query,
            retry_count=values.get("retry_count", 0),
            is_helpful=values.get("is_helpful", False),
            thread_id=tid,
            run_id=run_id,
        )
    except Exception as exc:
        return RunResult(query=query, retry_count=0, is_helpful=False,
                         thread_id=tid, run_id=run_id, error=str(exc))


async def run_suite(url: str, label: str) -> list[RunResult]:
    """Fire all queries in parallel, print results when all done."""
    print(f"\n{'═'*62}")
    print(f"  {label}")
    print(f"  {url}")
    print(f"  {len(TEST_QUERIES)} queries running in parallel …")
    print(f"{'═'*62}")

    client  = get_client(url=url)
    # All queries start at the same time; total wait ≈ slowest single query
    results = await asyncio.gather(*[run_query(client, q) for q in TEST_QUERIES])

    for r in results:
        short = textwrap.shorten(r.query, width=45, placeholder="…")
        if r.error:
            print(f"  ✗  {short:<47}  ERROR: {r.error}")
        else:
            tag = " ⟳ RETRIED" if r.looped else ""
            print(f"  ✓  {short:<47}  retry={r.retry_count}  helpful={str(r.is_helpful):<5}{tag}")

    return list(results)

## Local — `langgraph dev`
Make sure `uv run langgraph dev` is running in another terminal first.

In [7]:
local_results = await run_suite(LOCAL_URL, "LOCAL — langgraph dev")


══════════════════════════════════════════════════════════════
  LOCAL — langgraph dev
  http://localhost:55641
  3 queries running in parallel …
══════════════════════════════════════════════════════════════
  ✓  How often should I deworm my cat?                retry=1  helpful=True 
  ✓  What are signs of feline dehydration?            retry=1  helpful=True 
  ✓  My cat is sick, help.                            retry=1  helpful=True 


## Production — GCP Cloud Run

In [8]:
prod_results = await run_suite(PROD_URL, "PRODUCTION — GCP Cloud Run")


══════════════════════════════════════════════════════════════
  PRODUCTION — GCP Cloud Run
  https://cat-health-agent-316461491006.us-central1.run.app
  3 queries running in parallel …
══════════════════════════════════════════════════════════════
  ✓  How often should I deworm my cat?                retry=1  helpful=True 
  ✗  What are signs of feline dehydration?            ERROR: status=running after 180s
  ✓  My cat is sick, help.                            retry=1  helpful=True 


## Comparison table

In [9]:
print(f"\n{'═'*66}")
print("  Studio (local) vs. Production")
print(f"{'═'*66}")
print(f"  {'Query':<45}  {'Local':^14}  {'Prod':^14}  Match?")
print(f"  {'─'*45}  {'─'*14}  {'─'*14}  {'─'*6}")

diffs = []
for loc, pro in zip(local_results, prod_results):
    short    = textwrap.shorten(loc.query, width=43, placeholder="…")
    loc_cell = f"r={loc.retry_count} ok={int(loc.is_helpful)}" if not loc.error else "ERROR"
    pro_cell = f"r={pro.retry_count} ok={int(pro.is_helpful)}" if not pro.error else "ERROR"
    same     = (not loc.error and not pro.error
                and loc.is_helpful == pro.is_helpful
                and loc.retry_count == pro.retry_count)
    match    = "✓" if same else "✗ DIFF"
    if not same:
        diffs.append((loc.query, loc, pro))
    print(f"  {short:<45}  {loc_cell:^14}  {pro_cell:^14}  {match}")

print()
print("  r = retry_count (1 = passed first try, 2+ = retried)")
print("  ok = is_helpful (1 = passed, 0 = not helpful / exhausted)")

if diffs:
    print(f"\n  ⚠  {len(diffs)} query/ies behaved differently across environments.")
    print("     This is expected — live tool results and LLM sampling vary.")
    for q, l, p in diffs:
        print(f"     • {textwrap.shorten(q, 55)}")
        if not l.error: print(f"       local: retry={l.retry_count} helpful={l.is_helpful}")
        else:           print(f"       local: {l.error}")
        if not p.error: print(f"       prod:  retry={p.retry_count} helpful={p.is_helpful}")
        else:           print(f"       prod:  {p.error}")
else:
    print("\n  ✓ All queries matched (this run — may differ on re-run).")


══════════════════════════════════════════════════════════════════
  Studio (local) vs. Production
══════════════════════════════════════════════════════════════════
  Query                                              Local            Prod       Match?
  ─────────────────────────────────────────────  ──────────────  ──────────────  ──────
  How often should I deworm my cat?                 r=1 ok=1        r=1 ok=1     ✓
  What are signs of feline dehydration?             r=1 ok=1         ERROR       ✗ DIFF
  My cat is sick, help.                             r=1 ok=1        r=1 ok=1     ✓

  r = retry_count (1 = passed first try, 2+ = retried)
  ok = is_helpful (1 = passed, 0 = not helpful / exhausted)

  ⚠  1 query/ies behaved differently across environments.
     This is expected — live tool results and LLM sampling vary.
     • What are signs of feline dehydration?
       local: retry=1 helpful=True
       prod:  status=running after 180s


## LangSmith thread IDs
Use these to find the specific traces in [smith.langchain.com](https://smith.langchain.com) → project `cat-health-agent`.

In [10]:
for env_label, results in [("LOCAL", local_results), ("PROD", prod_results)]:
    print(f"── {env_label} ──")
    for r in results:
        short = textwrap.shorten(r.query, 42, placeholder="…")
        tag   = "⟳ RETRIED" if r.looped else ("ERROR    " if r.error else "✓ passed ")
        print(f"  {tag}  thread={r.thread_id[:8]}…  \"{short}\"")
        if r.error:
            print(f"            {r.error}")
    print()

── LOCAL ──
  ✓ passed   thread=019f3cbc…  "How often should I deworm my cat?"
  ✓ passed   thread=019f3cbc…  "What are signs of feline dehydration?"
  ✓ passed   thread=019f3cbc…  "My cat is sick, help."

── PROD ──
  ✓ passed   thread=019f3cbd…  "How often should I deworm my cat?"
  ERROR      thread=019f3cbd…  "What are signs of feline dehydration?"
            status=running after 180s
  ✓ passed   thread=019f3cbd…  "My cat is sick, help."



## What to look for in LangSmith

```
Pass (retry_count = 1)
  run_agent
  └── [inner ReAct: tool calls + final AIMessage]
  judge_response
  └── HelpfulnessVerdict { is_helpful: true, reason: "..." }

Retry (retry_count = 2+)
  run_agent          ← first attempt
  judge_response     ← is_helpful: false
  run_agent          ← second attempt (nudge injected)
  judge_response     ← is_helpful: true  (or retry cap hit)

Exhausted (retry_count = 3, is_helpful = False)
  Same as retry but judge_response appears 3×;
  loop exits by the retry_count cap, not a passing verdict.
```

**Studio vs Production:** same graph logic, but the number of retries
for the same query can differ — live tool results (Tavily, Arxiv) change
over time and LLM sampling adds variance even at temperature=0.